##Ingestion via Auto Loader

In [0]:
landing_volume_path = f"/Volumes/cdl/bronze/landing_files/"

# display(dbutils.fs.ls(landing_volume_path))

output_path = "abfss://destination-cdl@datalakedevesh.dfs.core.windows.net/bronze/auto_loader/"
checkpoint_location = "abfss://destination-cdl@datalakedevesh.dfs.core.windows.net/checkpoint/"
schema_location = "abfss://destination-cdl@datalakedevesh.dfs.core.windows.net/schemas/"

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","json")\
        .option("cloudFiles.schemaLocation",schema_location)\
            .load(landing_volume_path)

df.printSchema()

In [0]:
query = df.writeStream.\
    format("delta")\
        .option("checkpointLocation",checkpoint_location)\
            .trigger(availableNow=True)\
                .start(output_path)

query.awaitTermination()

In [0]:
#display(dbutils.fs.ls(checkpoint_location))

In [0]:
#display(dbutils.fs.ls(schema_location))

In [0]:
#display(spark.read.format("delta").load(output_path))

##specifics for each file type

In [0]:
df_call_activity = spark.readStream\
    .format("cloudFiles")\
        .option("cloudFiles.format","json")\
            .option("cloudFiles.schemaLocation",f"{schema_location}call_activity/")\
                .option("cloudFiles.schemaEvolutionMode","rescue")\
                    .option("rescuedDataColumn","_rescued_data")\
                        .option("pathGlobFilter","call_activity_*.json")\
                            .load(landing_volume_path)

df_call_activity.printSchema()

In [0]:
query = df_call_activity.writeStream\
    .format("delta")\
        .option("checkpointLocation",f"{checkpoint_location}call_activity/")\
            .option("mergeSchema","true")\
                .trigger(availableNow=True)\
                    .start(f"{output_path}call_activity/")

query.awaitTermination()


In [0]:
df = spark.read.format("delta").load(f"{output_path}call_activity")

# print(df.count())
display(df)

In [0]:
print("hello")